# LLM Policy Document Analysis — Refactored RAG

## 1. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


## 2. Set API key sebagai environment variable

```python
from google.colab import userdata
import os
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
```



In [4]:
import os
# Untuk Colab:
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

assert os.getenv("GOOGLE_API_KEY"), "GOOGLE_API_KEY belum diset."

## 3. Import modul refactor

In [5]:
from ingest import ingest_sources
from build_vectorstore import build_vectorstore, load_vectorstore
from rag_query import get_llm, build_rag_chain, query_rag, keyword_frequency
from analysis import (
    add_sentiment_columns,
    word_frequencies,
    plot_wordcloud,
    build_cooccurrence_network,
    visualize_network,
)

ModuleNotFoundError: No module named 'ingest'

## 4. Ingest PDF/CSV → chunks


In [ ]:
sources = {
    "Visi_Pemerintah": "UU_RPJPN_2045.pdf",
    "Regulasi_SDM": "UU_ASN_2023.pdf",
    "Landasan_Teori": "consensus_data.pdf",
    "Digital_Gov_Ranking_2025": "2025_Digital_Government_Ranking_Report.pdf",
    "Government_at_a_Glance_SEA": "Government at a Glance Southeast Asia 2025 Indonesia.pdf",
    "Doc_1338_37_6206": "1338-37-6206-1-10-20251128.pdf",
    "Scopus_Doc_1": "1-s2.0-S0740624X25000516-main.pdf",
    "Scopus_Doc_2": "1-s2.0-S0268401220309944-main.pdf",
    "Rulinawaty_2020": "121136_Rulinawaty_2020_E_R.pdf",
    "Realitas_Lapangan": "scraping_news.csv",
    "Scraping_Website": "scrapping_website.csv",
}

documents = ingest_sources(sources)
print("Total chunks:", len(documents))

## 5. Build/load FAISS vectorstore

In [ ]:
VECTORSTORE_DIR = "vectorstore_reformasi_2045"

vectorstore = build_vectorstore(documents, VECTORSTORE_DIR)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print("Vectorstore siap.")

Jika vectorstore sudah pernah dibuat dan tidak perlu rebuild:

In [ ]:
# vectorstore = load_vectorstore(VECTORSTORE_DIR)
# retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

## 6. RAG query

In [ ]:
llm = get_llm(model="gemini-2.5-flash", temperature=0)
rag_chain = build_rag_chain(retriever, llm)

question = "Apa visi reformasi 2045 dalam konteks transformasi digital birokrasi Indonesia?"
response = rag_chain.invoke(question)
print(response)

## 7. Analisis sentimen

In [ ]:
import pandas as pd

df = pd.read_csv("scraping_news.csv")
df = df[["text"]].dropna().drop_duplicates()

df = add_sentiment_columns(df)
display(df.head())
display(df["sentiment"].value_counts())

## 8. Word frequency dan word cloud

In [ ]:
top_words = word_frequencies(df["text"], top_n=20)
for word, count in top_words:
    print(f"{word}: {count}")

plot_wordcloud(
    df["text"],
    top_n=50,
    title="Word Cloud – Digital Reform Discourse",
)

## 9. Co-occurrence network

In [ ]:
G, word_freq = build_cooccurrence_network(
    df["text"],
    window_size=4,
    top_n=40,
    min_freq=2,
)

print("Node:", len(G.nodes()))
print("Edge:", len(G.edges()))

visualize_network(
    G,
    word_freq,
    title="Co-occurrence Network – RPJMN Discourse",
)

## 10. Keyword frequency

In [ ]:
keywords = ["AI", "Digital-First", "Korupsi", "Cybersecurity"]
freq_table = keyword_frequency(vectorstore, keywords)
display(freq_table)